# Navi Mumbai Street Annotations Analysis

Analysis of Label Studio annotations for Navi Mumbai street images.

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.nonparametric.smoothers_lowess import lowess

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Data Loading & Cleaning

In [ ]:
with open('../../labelstudio/export_226573_project-226573-at-2026-02-05-21-13-d63366f2.json', 'r') as f:
    data = json.load(f)

print(f"Total images: {len(data)}")

In [ ]:
def parse_annotation(task, annotation):
    """Parse a single annotation into a flat dict."""
    row = {
        'task_id': task['id'],
        'annotation_id': annotation['id'],
        'annotator_email': annotation['completed_by']['email'],
        'image': task['data']['image'],
        'file_upload': task.get('file_upload', ''),
    }
    
    for result in annotation['result']:
        field_name = result['from_name']
        if result['type'] == 'taxonomy':
            value = result['value']['taxonomy'][0][0] if result['value']['taxonomy'] else None
        elif result['type'] == 'choices':
            value = result['value']['choices'][0] if result['value']['choices'] else None
        elif result['type'] == 'textarea':
            value = result['value']['text'][0] if result['value']['text'] else None
        else:
            value = None
        row[field_name] = value
    
    return row

rows = []
for task in data:
    for annotation in task['annotations']:
        rows.append(parse_annotation(task, annotation))

df = pd.DataFrame(rows)
print(f"Total annotations: {len(df)}")

In [ ]:
core_fields = ['men_count', 'women_count', 'men_twowheeler', 'women_twowheeler', 
               'footpath', 'lane_markings', 'potholes', 'litter']
skip_fields = ['bus_station', 'railway_station', 'street_vendor']

def is_skip_row(row):
    """Check if row only has skip fields filled (and all are 'No')."""
    has_core = any(pd.notna(row.get(f)) for f in core_fields if f in row.index)
    has_skip = all(
        f in row.index and pd.notna(row.get(f)) and row.get(f) == 'No' 
        for f in skip_fields
    )
    return (not has_core) and has_skip

skip_mask = df.apply(is_skip_row, axis=1)
print(f"Skip rows (incomplete annotations): {skip_mask.sum()}")
print(f"Valid annotations: {(~skip_mask).sum()}")

df_valid = df[~skip_mask].copy()

In [ ]:
count_cols = ['men_count', 'women_count', 'men_twowheeler', 'women_twowheeler']

def convert_count(val):
    if pd.isna(val):
        return np.nan
    if val == '>10':
        return 11
    try:
        return int(val)
    except (ValueError, TypeError):
        return np.nan

for col in count_cols:
    if col in df_valid.columns:
        df_valid[col] = df_valid[col].apply(convert_count)

for col in count_cols:
    if col in df_valid.columns:
        df_valid[col] = df_valid[col].fillna(0)

print("NaN values treated as 0 (no people visible in that category)")
df_valid[count_cols].head(10)

In [ ]:
df_valid['total_pedestrians'] = df_valid['men_count'] + df_valid['women_count']
df_valid['total_twowheelers'] = df_valid['men_twowheeler'] + df_valid['women_twowheeler']
df_valid['total_women'] = df_valid['women_count'] + df_valid['women_twowheeler']
df_valid['total_men'] = df_valid['men_count'] + df_valid['men_twowheeler']
df_valid['total_people'] = df_valid['total_pedestrians'] + df_valid['total_twowheelers']

print(f"Images with 0 people: {(df_valid['total_people'] == 0).sum()}")
print(f"Images with people: {(df_valid['total_people'] > 0).sum()}")

## 2. Summary Statistics

In [ ]:
print(f"Total images: {df_valid['task_id'].nunique()}")
print(f"Total valid annotations: {len(df_valid)}")
print(f"Unique annotators: {df_valid['annotator_email'].nunique()}")

In [ ]:
multi_annotated = df_valid.groupby('task_id').size()
multi_annotated = multi_annotated[multi_annotated > 1]
print(f"Images with multiple annotations: {len(multi_annotated)}")
if len(multi_annotated) > 0:
    print(f"Annotation counts: {multi_annotated.value_counts().to_dict()}")

In [ ]:
annotator_counts = df_valid['annotator_email'].value_counts()
print("Annotations by annotator:")
print(annotator_counts)

In [ ]:
print("Count Statistics (after filling NaN with 0):")
print(df_valid[count_cols].describe().round(2))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, col in zip(axes.flat, count_cols):
    if col in df_valid.columns:
        data_col = df_valid[col]
        ax.hist(data_col, bins=range(0, 14), edgecolor='black', alpha=0.7)
        ax.set_xlabel(col)
        ax.set_ylabel('Frequency')
        ax.set_title(f'Distribution of {col}')
        ax.set_xticks(range(0, 13))
        ax.axvline(data_col.mean(), color='red', linestyle='--', label=f'Mean: {data_col.mean():.2f}')
        ax.legend()

plt.tight_layout()
plt.show()

## 3. Gender Analysis

### 3A. Proportion Women

Proportion women = women / (women + men)
- Only women present → 1.0
- Only men present → 0.0  
- 50-50 → 0.5
- No people → exclude (undefined)

In [ ]:
df_valid['prop_women_pedestrians'] = df_valid['women_count'] / (df_valid['men_count'] + df_valid['women_count']).replace(0, np.nan)
df_valid['prop_women_twowheelers'] = df_valid['women_twowheeler'] / (df_valid['men_twowheeler'] + df_valid['women_twowheeler']).replace(0, np.nan)
df_valid['prop_women'] = df_valid['total_women'] / df_valid['total_people'].replace(0, np.nan)

print("Proportion Women Statistics:")
print(f"\nOverall (all people):")
pw = df_valid['prop_women'].dropna()
print(f"  N: {len(pw)}")
print(f"  Mean: {pw.mean():.3f}, Median: {pw.median():.3f}, Std: {pw.std():.3f}")

print(f"\nPedestrians only:")
pwp = df_valid['prop_women_pedestrians'].dropna()
print(f"  N: {len(pwp)}")
print(f"  Mean: {pwp.mean():.3f}, Median: {pwp.median():.3f}, Std: {pwp.std():.3f}")

print(f"\nTwo-wheelers only:")
pwtw = df_valid['prop_women_twowheelers'].dropna()
print(f"  N: {len(pwtw)}")
print(f"  Mean: {pwtw.mean():.3f}, Median: {pwtw.median():.3f}, Std: {pwtw.std():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(pw, bins=20, edgecolor='black', alpha=0.7)
axes[0].axvline(0.5, color='green', linestyle=':', linewidth=2, label='Parity (0.5)')
axes[0].axvline(pw.median(), color='red', linestyle='--', label=f'Median: {pw.median():.3f}')
axes[0].set_xlabel('Proportion Women')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Overall')
axes[0].legend()

axes[1].hist(pwp, bins=20, edgecolor='black', alpha=0.7)
axes[1].axvline(0.5, color='green', linestyle=':', linewidth=2, label='Parity (0.5)')
axes[1].axvline(pwp.median(), color='red', linestyle='--', label=f'Median: {pwp.median():.3f}')
axes[1].set_xlabel('Proportion Women')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Pedestrians')
axes[1].legend()

axes[2].hist(pwtw, bins=20, edgecolor='black', alpha=0.7)
axes[2].axvline(0.5, color='green', linestyle=':', linewidth=2, label='Parity (0.5)')
axes[2].axvline(pwtw.median(), color='red', linestyle='--', label=f'Median: {pwtw.median():.3f}')
axes[2].set_xlabel('Proportion Women')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Two-wheelers')
axes[2].legend()

plt.tight_layout()
plt.show()

### 3B. Weighted vs Unweighted Proportion

- **Unweighted**: Mean of per-image proportions (each image counts equally)
- **Weighted**: Total women / Total people (each person counts equally)

In [ ]:
df_with_people = df_valid[df_valid['total_people'] > 0]

unweighted_prop_overall = df_with_people['prop_women'].mean()
weighted_prop_overall = df_with_people['total_women'].sum() / df_with_people['total_people'].sum()

print("Of all people observed, what proportion were women?")
print(f"\nOverall:")
print(f"  Unweighted (mean of image proportions): {unweighted_prop_overall:.3f}")
print(f"  Weighted (total women / total people): {weighted_prop_overall:.3f}")

df_with_pedestrians = df_valid[df_valid['total_pedestrians'] > 0]
weighted_prop_ped = df_with_pedestrians['women_count'].sum() / df_with_pedestrians['total_pedestrians'].sum()
unweighted_prop_ped = df_with_pedestrians['prop_women_pedestrians'].mean()

print(f"\nPedestrians:")
print(f"  Unweighted: {unweighted_prop_ped:.3f}")
print(f"  Weighted: {weighted_prop_ped:.3f}")

df_with_tw = df_valid[df_valid['total_twowheelers'] > 0]
weighted_prop_tw = df_with_tw['women_twowheeler'].sum() / df_with_tw['total_twowheelers'].sum()
unweighted_prop_tw = df_with_tw['prop_women_twowheelers'].mean()

print(f"\nTwo-wheelers:")
print(f"  Unweighted: {unweighted_prop_tw:.3f}")
print(f"  Weighted: {weighted_prop_tw:.3f}")

### 3C. LOWESS: Busyness vs Proportion Women

Does gender composition change with how busy a location is?

In [ ]:
print("Busyness Statistics (total people per image):")
tp = df_valid['total_people']
print(f"  Mean: {tp.mean():.2f}, Median: {tp.median():.2f}, Std: {tp.std():.2f}")
print(f"  Min: {tp.min():.0f}, Max: {tp.max():.0f}")

In [ ]:
valid_busy = df_valid[df_valid['total_people'] > 0][['total_people', 'prop_women']].dropna()
x = valid_busy['total_people'].values
y = valid_busy['prop_women'].values

lowess_result = lowess(y, x, frac=0.3)

fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(x, y, alpha=0.3, edgecolor='none', label='Data')
ax.plot(lowess_result[:, 0], lowess_result[:, 1], 'r-', linewidth=2, label='LOWESS')
ax.axhline(0.5, color='green', linestyle=':', linewidth=2, label='Parity (0.5)')
ax.axhline(y.mean(), color='blue', linestyle='--', alpha=0.7, label=f'Mean: {y.mean():.3f}')

ax.set_xlabel('Total People (Busyness)')
ax.set_ylabel('Proportion Women')
ax.set_title('Busyness vs Proportion Women')
ax.legend()
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

r, p = stats.pearsonr(x, y)
print(f"Pearson correlation: r={r:.3f}, p={p:.4f}")

### 3D. Sex Ratio vs Navi Mumbai Baseline

Sex ratio defined as females per 1000 males (Indian demographic convention). 

Navi Mumbai (Thane district urban) sex ratio: ~910 (Census 2011). Using 910 as baseline.

In [ ]:
NAVI_MUMBAI_BASELINE_SEX_RATIO = 910

df_valid['sex_ratio_pedestrians'] = (df_valid['women_count'] / df_valid['men_count'].replace(0, np.nan)) * 1000
df_valid['sex_ratio_twowheelers'] = (df_valid['women_twowheeler'] / df_valid['men_twowheeler'].replace(0, np.nan)) * 1000

print("Sex Ratio Statistics (females per 1000 males):")
print(f"Navi Mumbai baseline (approx): {NAVI_MUMBAI_BASELINE_SEX_RATIO}")
print(f"\nPedestrians:")
sr_ped = df_valid['sex_ratio_pedestrians'].dropna()
print(f"  N: {len(sr_ped)}")
print(f"  Mean: {sr_ped.mean():.1f}, Median: {sr_ped.median():.1f}, Std: {sr_ped.std():.1f}")
print(f"\nTwo-wheelers:")
sr_tw = df_valid['sex_ratio_twowheelers'].dropna()
print(f"  N: {len(sr_tw)}")
print(f"  Mean: {sr_tw.mean():.1f}, Median: {sr_tw.median():.1f}, Std: {sr_tw.std():.1f}")

weighted_sr_ped = (df_with_pedestrians['women_count'].sum() / df_with_pedestrians['men_count'].sum()) * 1000
weighted_sr_tw = (df_with_tw['women_twowheeler'].sum() / df_with_tw['men_twowheeler'].sum()) * 1000
print(f"\nWeighted sex ratios (total women / total men * 1000):")
print(f"  Pedestrians: {weighted_sr_ped:.1f}")
print(f"  Two-wheelers: {weighted_sr_tw:.1f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sr_ped_plot = sr_ped[sr_ped < 5000]
axes[0].hist(sr_ped_plot, bins=30, edgecolor='black', alpha=0.7)
axes[0].axvline(NAVI_MUMBAI_BASELINE_SEX_RATIO, color='blue', linestyle='-', linewidth=2, 
                label=f'Navi Mumbai baseline: {NAVI_MUMBAI_BASELINE_SEX_RATIO}')
axes[0].axvline(sr_ped_plot.median(), color='red', linestyle='--', 
                label=f'Median: {sr_ped_plot.median():.0f}')
axes[0].axvline(1000, color='green', linestyle=':', label='Parity (1000)')
axes[0].set_xlabel('Sex Ratio (females per 1000 males)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Sex Ratio Distribution (Pedestrians)')
axes[0].legend()

sr_tw_plot = sr_tw[sr_tw < 5000]
axes[1].hist(sr_tw_plot, bins=30, edgecolor='black', alpha=0.7)
axes[1].axvline(NAVI_MUMBAI_BASELINE_SEX_RATIO, color='blue', linestyle='-', linewidth=2,
                label=f'Navi Mumbai baseline: {NAVI_MUMBAI_BASELINE_SEX_RATIO}')
axes[1].axvline(sr_tw_plot.median(), color='red', linestyle='--',
                label=f'Median: {sr_tw_plot.median():.0f}')
axes[1].axvline(1000, color='green', linestyle=':', label='Parity (1000)')
axes[1].set_xlabel('Sex Ratio (females per 1000 males)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Sex Ratio Distribution (Two-wheelers)')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
print(f"One-sample t-test against Navi Mumbai baseline ({NAVI_MUMBAI_BASELINE_SEX_RATIO}):")

t_ped, p_ped = stats.ttest_1samp(sr_ped, NAVI_MUMBAI_BASELINE_SEX_RATIO)
print(f"\nPedestrians:")
print(f"  t-statistic: {t_ped:.3f}")
print(f"  p-value: {p_ped:.4f}")
print(f"  Significantly different from baseline: {'Yes' if p_ped < 0.05 else 'No'}")

t_tw, p_tw = stats.ttest_1samp(sr_tw, NAVI_MUMBAI_BASELINE_SEX_RATIO)
print(f"\nTwo-wheelers:")
print(f"  t-statistic: {t_tw:.3f}")
print(f"  p-value: {p_tw:.4f}")
print(f"  Significantly different from baseline: {'Yes' if p_tw < 0.05 else 'No'}")

## 4. Gender by POI (Points of Interest)

In [ ]:
poi_fields = ['bus_station', 'railway_station', 'street_vendor']

print("POI Image Counts:")
for poi in poi_fields:
    if poi in df_valid.columns:
        yes_count = (df_valid[poi] == 'Yes').sum()
        no_count = (df_valid[poi] == 'No').sum()
        print(f"  {poi}: Yes={yes_count}, No={no_count}")

In [ ]:
for poi in poi_fields:
    if poi in df_valid.columns:
        print(f"\n{'='*50}")
        print(f"{poi.upper()}")
        print(f"{'='*50}")
        
        yes_data = df_valid[(df_valid[poi] == 'Yes') & (df_valid['total_people'] > 0)]
        no_data = df_valid[(df_valid[poi] == 'No') & (df_valid['total_people'] > 0)]
        
        print(f"\nImages with people: Yes={len(yes_data)}, No={len(no_data)}")
        
        if len(yes_data) > 0:
            print(f"\nProportion Women (images with {poi}=Yes):")
            print(f"  Mean: {yes_data['prop_women'].mean():.3f}")
            print(f"  Median: {yes_data['prop_women'].median():.3f}")
            
        if len(no_data) > 0:
            print(f"\nProportion Women (images with {poi}=No):")
            print(f"  Mean: {no_data['prop_women'].mean():.3f}")
            print(f"  Median: {no_data['prop_women'].median():.3f}")
        
        if len(yes_data) >= 5 and len(no_data) >= 5:
            u_stat, p_val = stats.mannwhitneyu(
                yes_data['prop_women'].dropna(), 
                no_data['prop_women'].dropna(), 
                alternative='two-sided'
            )
            print(f"\nMann-Whitney U test: U={u_stat:.1f}, p={p_val:.4f}")

## 5. Inter-Annotator Agreement

In [ ]:
multi_task_ids = multi_annotated.index.tolist()

print(f"Only {len(multi_task_ids)} images have multiple annotators.")
print("Inter-annotator agreement cannot be meaningfully assessed with this sample size.")
print(f"\nPrimary annotator: {df_valid['annotator_email'].value_counts().idxmax()} ({df_valid['annotator_email'].value_counts().max()} annotations)")

if len(multi_task_ids) > 0:
    df_multi = df_valid[df_valid['task_id'].isin(multi_task_ids)].copy()
    print(f"\nTotal annotations for multi-annotated images: {len(df_multi)}")
    print(f"\nFor reference, agreement on those {len(multi_task_ids)} images:")
    for col in count_cols:
        if col in df_multi.columns:
            agreement_data = []
            for task_id in multi_task_ids:
                task_annots = df_multi[df_multi['task_id'] == task_id][col]
                if len(task_annots) >= 2:
                    diff = task_annots.max() - task_annots.min()
                    agreement_data.append(diff)
            if agreement_data:
                exact_match = sum(1 for d in agreement_data if d == 0)
                close_match = sum(1 for d in agreement_data if d <= 1)
                print(f"  {col}: Exact match {exact_match}/{len(agreement_data)}, Within 1: {close_match}/{len(agreement_data)}")

## 6. Summary

In [ ]:
print("=" * 60)
print("SUMMARY")
print("=" * 60)

print(f"\nDataset Overview:")
print(f"  Total images: {df_valid['task_id'].nunique()}")
print(f"  Total valid annotations: {len(df_valid)}")
print(f"  Skipped (incomplete) annotations: {skip_mask.sum()}")
print(f"  Images with 0 people visible: {(df_valid['total_people'] == 0).sum()}")
print(f"  Unique annotators: {df_valid['annotator_email'].nunique()}")

print(f"\nProportion Women:")
print(f"  Unweighted (mean of image proportions): {unweighted_prop_overall:.3f}")
print(f"  Weighted (total women / total people): {weighted_prop_overall:.3f}")
print(f"  Pedestrians (weighted): {weighted_prop_ped:.3f}")
print(f"  Two-wheelers (weighted): {weighted_prop_tw:.3f}")

print(f"\nSex Ratio (females per 1000 males, Navi Mumbai baseline: ~{NAVI_MUMBAI_BASELINE_SEX_RATIO}):")
print(f"  Pedestrians (weighted): {weighted_sr_ped:.1f}")
print(f"  Two-wheelers (weighted): {weighted_sr_tw:.1f}")

print(f"\nBusyness:")
print(f"  Total people - Mean: {df_valid['total_people'].mean():.2f}, Median: {df_valid['total_people'].median():.2f}")

print(f"\nPOI Effects on Proportion Women:")
for poi in poi_fields:
    if poi in df_valid.columns:
        yes_data = df_valid[(df_valid[poi] == 'Yes') & (df_valid['total_people'] > 0)]
        no_data = df_valid[(df_valid[poi] == 'No') & (df_valid['total_people'] > 0)]
        if len(yes_data) > 0 and len(no_data) > 0:
            diff = yes_data['prop_women'].mean() - no_data['prop_women'].mean()
            print(f"  {poi}: Yes={yes_data['prop_women'].mean():.3f} vs No={no_data['prop_women'].mean():.3f} (diff={diff:+.3f})")